# Anatomy of LLMs — Hands-on Exercise

**Module:** M1 — LLM Fundamentals  
**Lesson:** L1 — Anatomy of LLMs  
**Audience:** AI Engineer (developer track)  
**Time:** ~114 min  
**Prereqs:** Python 3.10+, Anthropic API key  

## Why this exercise

You're an AI engineer who needs to understand the machinery you're building on top of. You'll inspect how LLMs tokenize input, experiment with generation parameters, and calculate real production costs — building intuition for how architecture choices affect the output your applications produce.

## Success criteria

You're done when:

- You have code that tokenizes 5 different inputs and compares token counts across two tokenizers
- You've called the Anthropic API with at least 4 different parameter configurations and can explain why each produces different output
- You've documented which parameter has the most impact on output quality for factual vs creative tasks, and why
- You can explain to a peer why multilingual text produces more tokens than English text of similar semantic length
- You've calculated per-call API costs and computed the multilingual cost multiplier
- You've estimated a context budget breakdown for a production application
- Your capstone capability document identifies at least 3 LLM capabilities your project needs, with parameter rationale and cost estimates

## Setup

Run this cell once. It installs the dependencies and reads your API keys from Colab Secrets (or local env vars).

- In Colab: open the key icon in the left sidebar and add `ANTHROPIC_API_KEY`.
- Locally: `export ANTHROPIC_API_KEY=...` before launching Jupyter.

**Never paste an API key into a cell.** This notebook will be pushed to GitHub at the end of the course as portfolio evidence.

In [9]:
%pip install -q anthropic tiktoken

import os

try:
    from google.colab import userdata  # Colab
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

assert ANTHROPIC_API_KEY, 'Set ANTHROPIC_API_KEY in Colab Secrets or your shell env.'

# Model tier switch — see shared/cheaper-model-substitution.md
MODEL_TIER = os.environ.get('MODEL_TIER', 'cheap')
MODEL = {
    'cheap':    'claude-haiku-4-5',
    'standard': 'claude-sonnet-5',
    'premium':  'claude-opus-4-8',
    'supreme':  'claude-fable-5',
}[MODEL_TIER]
print(f'Using model: {MODEL} (tier={MODEL_TIER})')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 999.8/999.8 kB 18.3 MB/s eta 0:00:00
Using model: claude-haiku-4-5 (tier=cheap)


---

## Part 1 — Tokenization Exploration (~35 min)

### Step 1 — Tokenize 5 inputs with tiktoken

Tokenize 5 different input types using tiktoken (`cl100k_base` encoding). For each, print the input, character count, token count, token-to-character ratio, and the decoded subword splits.

Inputs to tokenize:
1. **Plain English sentence**
2. **Python function** — at least 5 lines of real code
3. **Non-Latin script** — Hebrew, Arabic, Chinese, or Japanese
4. **JSON object** — nested, at least 3 fields
5. **Long number sequence** — a UUID or similar

In [10]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Pre-filled: the 5 input types you will tokenize
inputs = [
    ("English", "The quick brown fox jumps over the lazy dog."),
    ("Python", "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)"),
    ("Hebrew", "הבינה המלאכותית משנה את עולם הפיתוח"),
    ("JSON", '{"user": {"name": "Alice", "role": "engineer", "projects": ["api", "ml-pipeline"]}}'),
    ("UUID", "550e8400-e29b-41d4-a716-446655440000"),
]

for label, text in inputs:
    # TODO: encode `text` with enc.encode() → a list of integer token IDs
    tokens = enc.encode(text)
    print(tokens)

    # TODO: decode each token ID individually → a list of subword strings
    #   Hint: enc.decode([t]) decodes a single token; apply it to each element of tokens
    decoded = [enc.decode([t]) for t in tokens]

    print(f"\n--- {label} ---")
    print(f"Input: {text[:80]}{'...' if len(text) > 80 else ''}")
    print(f"Characters: {len(text)}")
    print(f"Tokens: {len(tokens)}")
    print(f"Tokens to Character: {len(tokens)/len(text):.3f}")
    print(f"Decoded: {decoded}")

[791, 4062, 14198, 39935, 35308, 927, 279, 16053, 5679, 13]

--- English ---
Input: The quick brown fox jumps over the lazy dog.
Characters: 44
Tokens: 10
Tokens to Character: 0.227
Decoded: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']
[755, 76798, 1471, 997, 262, 422, 308, 2717, 220, 16, 512, 286, 471, 308, 198, 262, 471, 76798, 1471, 12, 16, 8, 489, 76798, 1471, 12, 17, 8]

--- Python ---
Input: def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fi...
Characters: 92
Tokens: 28
Tokens to Character: 0.304
Decoded: ['def', ' fibonacci', '(n', '):\n', '   ', ' if', ' n', ' <=', ' ', '1', ':\n', '       ', ' return', ' n', '\n', '   ', ' return', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']
[47071, 76625, 43336, 254, 47071, 70446, 68406, 50391, 59610, 147, 249, 37769, 103, 43336, 103, 92611, 59511, 95526, 47071, 63060, 55614, 17732, 95, 37769, 250, 147, 251, 70446, 147, 97, 43336, 103, 37769,

### Step 2 — Compare with Anthropic's tokenizer

Use `client.messages.count_tokens()` to count tokens for the same 5 inputs through Claude's tokenizer. Compare the results with tiktoken — different tokenizer vocabularies produce different counts.

In [11]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Tokenizer comparison: tiktoken (cl100k_base) vs Anthropic\n")
for label, text in inputs:
    tiktoken_count = len(enc.encode(text))

    # Call client.messages.count_tokens() to count tokens via Anthropic
    response = client.messages.count_tokens(
        model=MODEL,
        messages=[{"role": "user", "content": text}]
    )
    anthropic_count = response.input_tokens

    # Compute diff
    diff = anthropic_count - tiktoken_count

    # TODO: print one line per label:
    print(f"{label:8s}  tiktoken={tiktoken_count:4d}  anthropic={anthropic_count:4d}  diff={diff:+d}")

Tokenizer comparison: tiktoken (cl100k_base) vs Anthropic

English   tiktoken=  10  anthropic=  18  diff=+8
Python    tiktoken=  28  anthropic=  40  diff=+12
Hebrew    tiktoken=  34  anthropic=  30  diff=-4
JSON      tiktoken=  28  anthropic=  34  diff=+6
UUID      tiktoken=  18  anthropic=  28  diff=+10


### Step 3 — Tokenization observations

Answer in your own words (edit this cell):

1. Which input type produced the most tokens relative to its character count? Why?<br />
Hebrew - because it's a non-Latin language so it uses a lot of tokens. It is a less familiar language. Tokens to Character: 0.971
2. How did the token counts differ between tiktoken and Anthropic's tokenizer?<br />
For all of them, Anthropic used more besides for Hebrew
3. What does this mean for cost estimation in a multilingual application?<br />
Hebrew will be more expensive
4. If your capstone processes non-English text, how would you adjust your cost estimates?<br />
It would have to be a higher estimate

---

## Part 2 — Generation Parameter Experiments (~45 min)

### Step 4 — Experiment helper

Run this cell to define a reusable function for parameter experiments. Read the function — you'll use it for the next 4 experiments.

In [22]:
def run_experiment(
    prompt: str,
    system: str = "",
    temperature: float = 1.0,
    max_tokens: int = 200,
    stop_sequences: list[str] | None = None,
    runs: int = 1,
) -> None:
    for i in range(runs):
        kwargs = {
            "model": MODEL,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        }
        if system:
            kwargs["system"] = system
        if stop_sequences:
            kwargs["stop_sequences"] = stop_sequences

        response = client.messages.create(**kwargs)
        text = response.content[0].text
        usage = response.usage

        prefix = f"  Run {i+1}: " if runs > 1 else "  Response: "
        print(f"{prefix}{text}")
        print(f"    [tokens: in={usage.input_tokens}, out={usage.output_tokens}]")
    print()

### Step 5 — Experiment A: Temperature on a factual task

Run the same factual prompt at temperature 0.0, 0.5, and 1.0 (3 times each). Watch: does the answer change? Does the format change?

In [13]:
print("=== Experiment A: Temperature on factual task ===")

# TODO: iterate over temperatures [0.0, 0.5, 1.0]
#   For each temperature:
#     print(f"\nTemperature {temp}:")
#     call run_experiment() with:
#       prompt="What is the capital of France? Answer in one word."
#       temperature=temp
#       runs=3
#   Observe: does the answer change? Does the format change?

temperatures = [0.0, 0.5, 1.0]
for temp in temperatures:
    print(f"\nTemperature {temp}:")
    run_experiment(
        prompt="What is the capital of France? Answer in one word.",
        temperature=temp,
        runs=3
    )

=== Experiment A: Temperature on factual task ===

Temperature 0.0:
  Run 1: Paris
    [tokens: in=19, out=4]
  Run 2: Paris
    [tokens: in=19, out=4]
  Run 3: Paris
    [tokens: in=19, out=4]


Temperature 0.5:
  Run 1: Paris
    [tokens: in=19, out=4]
  Run 2: Paris
    [tokens: in=19, out=4]
  Run 3: Paris
    [tokens: in=19, out=4]


Temperature 1.0:
  Run 1: Paris
    [tokens: in=19, out=4]
  Run 2: Paris
    [tokens: in=19, out=4]
  Run 3: Paris
    [tokens: in=19, out=4]



### Step 6 — Experiment B: Temperature on a creative task

Same temperature sweep, but on a creative prompt. How does variety change? Is higher temperature always "more creative"?

In [14]:
print("=== Experiment B: Temperature on creative task ===")

# TODO: same loop structure as Experiment A
#   prompt="Write a two-line poem about debugging."
#   temperature in [0.0, 0.5, 1.0], runs=3
#   Observe: how does output variety change? Is higher temperature always "more creative"?

temperatures = [0.0, 0.5, 1.0]
for temp in temperatures:
    print(f"\nTemperature {temp}:")
    run_experiment(
        prompt="Write a two-line poem about debugging.",
        temperature=temp,
        runs=3
    )


=== Experiment B: Temperature on creative task ===

Temperature 0.0:
  Run 1: # Debugging

A thousand lines of code run wild and wrong,
One missing semicolon—found at last, I'm strong.
    [tokens: in=16, out=32]
  Run 2: # Debugging

A thousand lines of code run wild and free,
Till one small semicolon sets logic free.
    [tokens: in=16, out=28]
  Run 3: # Debugging

A thousand lines of code run wild and wrong,
One missing semicolon—found at last, I'm strong.
    [tokens: in=16, out=32]


Temperature 0.5:
  Run 1: # Debug

A tangled web of logic gone astray,
One semicolon found—the light of day.
    [tokens: in=16, out=28]
  Run 2: # Debugging

A thousand lines of code run wild and free,
Till one small typo sets the logic free.
    [tokens: in=16, out=28]
  Run 3: # Debugging

A thousand lines of code run wild and wrong,
One missing semicolon—found at last, relief so strong.
    [tokens: in=16, out=32]


Temperature 1.0:
  Run 1: # Debugging

A thousand lines of code gone wrong,
One s

### Step 7 — Experiment C: System prompt effect

Same user prompt, two very different system prompts. How much does the system prompt control output style, length, and tone?

In [15]:
print("=== Experiment C: System prompt effect ===")
prompt = "Explain what a REST API is."

print(f"\nSystem: concise technical writer")
run_experiment(
    system="You are a concise technical writer. Be brief.",
    prompt="Explain what a REST API is.",
    temperature=0.5,
    runs=3
)

print(f"\nSystem: friendly, detailed explainer")
run_experiment(
    system="You are a friendly, detailed explainer who uses analogies and examples from everyday life.",
    prompt="Explain what a REST API is.",
    temperature=0.5,
    runs=3
)

# TODO: call run_experiment() twice with the same `prompt` but different system args:
#   Call 1 — system="You are a concise technical writer. Be brief."
#   Call 2 — system="You are a friendly, detailed explainer who uses analogies and examples from everyday life."
# Print a descriptive label (e.g. print("System: concise technical writer")) before each call

=== Experiment C: System prompt effect ===

System: concise technical writer
  Run 1: # REST API

A **REST API** (Representational State Transfer) is a web service that allows applications to communicate over HTTP using standard operations.

## Key Principles

- **Resources**: Data is organized as resources (users, posts, products) identified by URLs
- **HTTP Methods**: Operations use standard verbs:
  - `GET` - retrieve data
  - `POST` - create data
  - `PUT` - update data
  - `DELETE` - remove data
- **Stateless**: Each request is independent; server doesn't store client context
- **JSON/XML**: Data is typically exchanged in JSON or XML format

## Example

```
GET /api/users/123          → Retrieve user with ID 123
POST /api/users             → Create a new user
PUT /api/users/123          → Update user 123
DELETE /api/users/123       → Delete user 
    [tokens: in=26, out=200]
  Run 2: # REST API

A **REST API** (Representational State Transfer) is an architectural style for buildin

### Step 8 — Experiment D: Stop sequences

Use `stop_sequences` to cut off generation early. Does the model stop where you expect? What happens if the stop sequence doesn't match the model's formatting exactly?

In [16]:
print("=== Experiment D: Stop sequences ===")

# TODO: call run_experiment() twice with prompt="List 5 programming languages:"
#   and temperature=0.0
#   Call 1 — no stop_sequences argument (let the model finish)
#   Call 2 — stop_sequences=["\n4."]
# Print a label before each call. Does the model stop where you expected?

print("\nCall 1: No stop sequences (let model finish)")
run_experiment(
    prompt="List 5 programming languages:",
    temperature=0.0
)

print("Call 2: Stop at '\\n4.' (should stop before item 4)")
run_experiment(
    prompt="List 5 programming languages:",
    temperature=0.0,
    stop_sequences=["\n4."]
)

=== Experiment D: Stop sequences ===

Call 1: No stop sequences (let model finish)
  Response: # 5 Programming Languages

1. **Python** - Known for simplicity and readability; widely used in data science, web development, and automation
2. **JavaScript** - The primary language for web development; runs in browsers and on servers (Node.js)
3. **Java** - A versatile, object-oriented language commonly used in enterprise applications and Android development
4. **C++** - A powerful language used for system software, game development, and performance-critical applications
5. **SQL** - A specialized language for managing and querying databases
    [tokens: in=14, out=123]

Call 2: Stop at '\n4.' (should stop before item 4)
  Response: # 5 Programming Languages

1. **Python** - Known for simplicity and readability; widely used in data science, web development, and automation
2. **JavaScript** - The primary language for web browsers; essential for front-end development
3. **Java** - A versatile

### Step 9 — Parameter observations

Answer in your own words (edit this cell):

1. Which parameter had the biggest impact on output quality for factual tasks? <br/>
For Factual tasks the temperature <br/> For creative tasks it's the system prompt and also the temperature, but mainly the system.<br/>
2. When would you use temperature 0 vs temperature 1 in a production application? Give a concrete example for each.<br/>
Temperature 0 - I would use it when I want the same factual response each time. For example an api response or the answer to how much something will cost in the store?<br/>
Temperature 1 - When writing something that needs a more creative answer like writing a blog post on LinkedIn, we want something more creative than just a dry, factual answer.<br/>
3. How powerful is the system prompt for controlling output? What are its limits? <br/>Very powerful, it can determine the tone and voice of the response.<br/>Limits - it's limited in not going against what it was trained on, so if it was trained to not talk about certain sensitive topics, the system prompt will not be able to override that.<br/>
4. What did you learn about stop sequences that would matter for parsing LLM output in production?<br/>If you want to print out a certain response and you have a stop sequence it will stop after a number of lines, resulting in an inaccurate response. If it's not necessary to have so many words, like if you want a summary of the news, then you can use a stop sequence.<br>
5. What surprised you most across all 4 experiments? <br/>
I was surprised that the system prompt and the various parameters such as temperature and stop sequence make such a difference in the response.

---

## Part 3 — Context Cost Calculations (~20 min)

**Anthropic API pricing reference** (volatile layer — verify at docs.anthropic.com before production use):

| Model | Tier | Input (per 1M tokens) | Output (per 1M tokens) |
|---|---|---|---|
| Claude Haiku 4.5 | `cheap` | $1.00 | $5.00 |
| Claude Sonnet 5 | `standard` | $3.00* | $15.00* |
| Claude Opus 4.8 | `premium` | $5.00 | $25.00 |
| Claude Fable 5 | `supreme` | $10.00 | $50.00 |

\* Sonnet 5 has introductory pricing of $2.00 / $10.00 through 2026-08-31. Sonnet 5 also uses a new tokenizer that produces ~35% more tokens than Sonnet 4.6 for the same text — factor that in when comparing token counts or costs across model generations.

### Step 10 — Per-call cost estimation

Using your token counts from Part 1, calculate the input cost for each of your 5 inputs on your currently configured model (`MODEL`, set by `MODEL_TIER` in Setup). Assume a 200-token output per call.

In [20]:
PRICING = {  # (input $/1M, output $/1M) — verify at docs.anthropic.com before production use
    'claude-haiku-4-5': (1.00, 5.00),
    'claude-sonnet-5':  (3.00, 15.00),
    'claude-opus-4-8':  (5.00, 25.00),
    'claude-fable-5':   (10.00, 50.00),
}
INPUT_PER_TOKEN, OUTPUT_PER_TOKEN = (p / 1_000_000 for p in PRICING[MODEL])

print(f"Per-call cost estimates ({MODEL}, 200-token output):\n")

for label, text in inputs:
    input_tokens = len(enc.encode(text))
    output_tokens = 200
    cost = (input_tokens * INPUT_PER_TOKEN) + (output_tokens * OUTPUT_PER_TOKEN)
    print(f"{label:8s}: {input_tokens:4d} in + {output_tokens} out = ${cost:.6f} per call")

# TODO: for each (label, text) in inputs:
#   input_tokens = len(enc.encode(text))
#   output_tokens = 200  (fixed assumption)
#   cost = (input_tokens * INPUT_PER_TOKEN) + (output_tokens * OUTPUT_PER_TOKEN)
#   print: f"{label:8s}: {input_tokens:4d} in + {output_tokens} out = ${cost:.6f} per call"

Per-call cost estimates (claude-haiku-4-5, 200-token output):

English :   10 in + 200 out = $0.001010 per call
Python  :   28 in + 200 out = $0.001028 per call
Hebrew  :   34 in + 200 out = $0.001034 per call
JSON    :   28 in + 200 out = $0.001028 per call
UUID    :   18 in + 200 out = $0.001018 per call


### Step 11 — Multilingual cost multiplier & context budget

Calculate the cost difference between English and non-Latin text. Then estimate how a production application divides a 200K context window.

In [18]:


import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

inputs = [
    ("English", "The quick brown fox jumps over the lazy dog."),
    ("Python", "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)"),
    ("Hebrew", "הבינה המלאכותית משנה את עולם הפיתוח"),
    ("JSON", '{"user": {"name": "Alice", "role": "engineer", "projects": ["api", "ml-pipeline"]}}'),
    ("UUID", "550e8400-e29b-41d4-a716-446655440000"),
]

def count_tokens(text):
    resp = client.messages.count_tokens(
        model="claude-sonnet-4-6",
        messages=[{"role": "user", "content": text}]
    )
    return resp.input_tokens

eng_tokens = count_tokens(inputs[0][1])
heb_tokens = count_tokens(inputs[2][1])

ratio = heb_tokens / eng_tokens
print(f"English tokens: {eng_tokens}")
print(f"Hebrew tokens:  {heb_tokens}")
print(f"Token ratio:    {ratio:.1f}x")
print(f"Cost multiplier: processing Hebrew is ~{ratio:.1f}x more expensive per semantic unit")

def daily_cost(input_tokens):
    return 10_000 * (input_tokens * INPUT_PER_TOKEN + 200 * OUTPUT_PER_TOKEN)

print(f"\nEnglish daily cost: ${daily_cost(eng_tokens):,.2f}")
print(f"Hebrew daily cost:  ${daily_cost(heb_tokens):,.2f}")



# TODO: get token counts for English (inputs[0]) and Hebrew (inputs[2])
#eng_tokens = ...
#heb_tokens = ...

# TODO: compute ratio = heb_tokens / eng_tokens and print:
#   f"English tokens: {eng_tokens}"
#   f"Hebrew tokens:  {heb_tokens}"
#   f"Token ratio:    {ratio:.1f}x"
#   f"Cost multiplier: processing Hebrew is ~{ratio:.1f}x more expensive per semantic unit"

# TODO: compute daily cost for 10_000 requests at the English and Hebrew token counts
#   daily = 10_000 * (input_tokens * INPUT_PER_TOKEN + 200 * OUTPUT_PER_TOKEN)
#   print both English and Hebrew daily costs

# Pre-filled: context budget template — replace "___" with your capstone's token estimates
#print("\nContext budget template (fill in for your capstone):")


English tokens: 18
Hebrew tokens:  30
Token ratio:    1.7x
Cost multiplier: processing Hebrew is ~1.7x more expensive per semantic unit

English daily cost: $10.18
Hebrew daily cost:  $10.30


## Your turn

**Capstone capability mapping** (~14 min)

Create a structured analysis of what your capstone project needs from an LLM. Be concrete:

1. List at least 3 LLM capabilities your project requires (generation, classification, extraction, summarization, reasoning, code generation, translation, etc.)<br />
a) Extraction - Reads a Fathom meeting transcript and extracts structured workflow definitions — triggers, actions, conditions, platform references, and ambiguities — into a JSON object that drives everything downstream.<br />
b) Build planning/Reasoning - Takes the extracted workflow JSON and reasons about dependency ordering, platform selection, conflict detection (e.g., "this segment already exists — reuse or create new?"), and produces an ordered, executable build plan with tool assignments per step.<br />
c) Code/Payload Generation - For each step in the build plan, generates the exact API payload — Make scenario JSON structure, MailerLite subscriber upsert body, WordPress webhook registration config — ready to execute as a tool call.<br />
d) Error - When an API call fails, reads the error response, classifies the failure type (wrong field type, resource doesn't exist, auth issue, rate limit), and generates a targeted fix — a corrected payload or a prerequisite step to insert — without restarting the full build.<br />
3) Summarization - After the build completes and a test payload has been sent through the full chain, generates a plain-language report for the developer: what was built, what was reused, what succeeded, what needs manual attention, with direct links to each created resource.<br />
2. For each capability: which generation parameters would you start with and why?<br />
| Capability | Temperature | Max Tokens | Why |
|:---|:---:|:---:|:---|
| **Extraction** | `0.0` | `1,000` | Deterministic JSON output — wrong extraction breaks everything downstream; zero room for variation |
| **Build Planning** | `0.1` | `800` | Near-deterministic; tiny flexibility to weigh equivalent approaches; CoT reasoning trace needs token room |
| **Payload Generation** | `0.0` | `600` | API payloads are exact specs — any variation produces malformed requests; one payload per call |
| **Error Recovery** | `0.1` | `300` | Slight flexibility for ambiguous errors; output is minimal — just the corrected field or missing step |
| **Verification Summary** | `0.3` | `500` | Only step where natural readable phrasing matters; input is structured so hallucination risk is low |1)
3. For each: which model tier (`cheap`/Haiku for fast and simple tasks, `standard`/Sonnet for balanced everyday work, `premium`/Opus for complex reasoning, `supreme`/Fable for the hardest long-horizon agentic work) and why?<br />
| Capability | Model Tier | Model | Why |
|:---|:---:|:---:|:---|
| **Extraction** | `standard` | Sonnet | Requires inference from messy conversational language — Haiku misses implicit references, Opus is overkill for constrained extraction |
| **Build Planning** | `premium` | Opus | Hardest reasoning step: dependency ordering, conflict detection, multi-platform sequencing — wrong order causes cascading failures; runs once per build so cost is justified |
| **Payload Generation** | `standard` | Sonnet | Well-scoped per step with RAG schema injected; Sonnet follows structured generation reliably — Haiku produces subtle field errors on nested JSON like Make scenarios |
| **Error Recovery** | `cheap` | Haiku | API error message tells you almost exactly what's wrong; this is classification + small correction, not reasoning — cost compounds across multiple retries so right-sizing matters |
| **Verification Summary** | `cheap` | Haiku | Pure summarization from structured build log data — all facts are provided, model just formats them; no reasoning needed |
4. Fill in the context budget template above with your capstone's estimates<br />
## Context Budget per Build

| Component | Tokens |
|:---|---:|
| System prompt (agent role, rules, tools manifest) | ~1,200 |
| Fathom transcript (avg 30-min meeting) | ~3,500 |
| Extracted workflow JSON (carried through all steps) | ~800 |
| RAG: retrieved API schema for current step | ~1,500 |
| RAG: error pattern matches (recovery steps only) | ~600 |
| Accumulated state (resource IDs, webhook URLs, build log) | ~500 |
| Current API response (tool result) | ~300 |
| **Heaviest single call input (planning step)** | **~6,000** |
| **Average per-step input (execution steps)** | **~4,100** |
| Output per step (avg) | ~500 |
| **Total tokens across full build (8 steps avg)** | **~38,000** |

5. Calculate your monthly cost estimate at your expected request volume

## Monthly Cost Estimate

**Assumptions:** 3 early clients = ~40 builds/month

| Step | Model | Calls/month | Avg Tokens/call | Est. Cost |
|:---|:---:|:---:|:---:|---:|
| Transcript extraction | Sonnet | 40 | ~5,000 | ~$3.00 |
| Build planning | Opus | 40 | ~7,000 | ~$21.00 |
| Payload generation (8 steps avg) | Sonnet | 320 | ~4,500 | ~$20.00 |
| Error recovery (avg 2× per build) | Haiku | 80 | ~2,000 | ~$0.50 |
| Verification summary | Haiku | 40 | ~2,500 | ~$0.30 |
| **Total** | | | | **~$45/month** |


In [21]:
capstone = {
    "project": "Meeting-to-Workflow Builder: reads Fathom transcripts and auto-deploys Make scenarios, MailerLite lists, and WordPress webhooks",
    "capabilities": [
        {
            "name": "extraction",
            "use_case": "Parse Fathom meeting transcript into structured workflow definitions — triggers, actions, conditions, platform references",
            "temperature": 0.0,
            "model_tier": "standard",
            "reasoning": "Deterministic JSON output required — wrong extraction breaks all downstream steps; Sonnet needed for inferring implicit intent from conversational language that Haiku misses",
        },
        {
            "name": "build_planning",
            "use_case": "Reason about dependency ordering, conflict detection, and platform sequencing to produce an executable build plan",
            "temperature": 0.1,
            "model_tier": "premium",
            "reasoning": "Hardest reasoning step — must understand that MailerLite segment must exist before Make scenario can reference its ID; wrong ordering causes cascading failures; runs once per build so Opus cost is justified",
        },
        {
            "name": "payload_generation",
            "use_case": "Generate exact API payloads for each build step — Make scenario JSON, MailerLite subscriber upsert, WordPress webhook config",
            "temperature": 0.0,
            "model_tier": "standard",
            "reasoning": "API payloads must be exact; RAG-retrieved schema constrains the output so Opus not needed; Sonnet follows structured generation reliably where Haiku produces subtle field-name errors on nested Make scenario JSON",
        },
        {
            "name": "error_recovery",
            "use_case": "Classify API failure type and generate a targeted fix — corrected payload or missing prerequisite step — without restarting the full build",
            "temperature": 0.1,
            "model_tier": "cheap",
            "reasoning": "API error message tells you almost exactly what's wrong; this is classification + small correction not reasoning; runs multiple times per build so right-sizing is critical for cost",
        },
        {
            "name": "verification_summary",
            "use_case": "Generate plain-language report of what was built, reused, succeeded, or needs manual attention, with links to each created resource",
            "temperature": 0.3,
            "model_tier": "cheap",
            "reasoning": "Pure summarization from structured build log — all facts provided, model just formats them readably; higher temp produces natural phrasing; no reasoning needed so Haiku is correct tier",
        },
    ],
    "daily_requests": 2,  # ~40 builds/month across 3 clients = ~1.3/day; using 2 as realistic daily average with headroom
}

print(f"Project: {capstone['project']}\n")
for cap in capstone["capabilities"]:
    print(f"{cap['name']}:")
    print(f"  Use case:    {cap['use_case']}")
    print(f"  Temperature: {cap['temperature']}")
    print(f"  Model tier:  {cap['model_tier']}")
    print(f"  Reasoning:   {cap['reasoning']}")
    print()

# Weighted average across all 5 capability types per full build
# Heaviest calls: planning (7,000 tokens in) + payload×8 (4,500 avg) + extraction (5,000)
# Lighter calls: error recovery (2,000) + summary (2,500)
# Weighted average across ~12 total calls per build:
avg_input_tokens = 4200   # weighted average input across all call types per build
avg_output_tokens = 500   # weighted average output per call

daily = capstone["daily_requests"]
print(f"Monthly cost estimates ({daily} req/day):")
for model_id, (in_price, out_price) in PRICING.items():
    monthly = daily * 30 * (avg_input_tokens * in_price / 1e6 + avg_output_tokens * out_price / 1e6)
    print(f"  {model_id:20s} ${monthly:.2f}")

Project: Meeting-to-Workflow Builder: reads Fathom transcripts and auto-deploys Make scenarios, MailerLite lists, and WordPress webhooks

extraction:
  Use case:    Parse Fathom meeting transcript into structured workflow definitions — triggers, actions, conditions, platform references
  Temperature: 0.0
  Model tier:  standard
  Reasoning:   Deterministic JSON output required — wrong extraction breaks all downstream steps; Sonnet needed for inferring implicit intent from conversational language that Haiku misses

build_planning:
  Use case:    Reason about dependency ordering, conflict detection, and platform sequencing to produce an executable build plan
  Temperature: 0.1
  Model tier:  premium
  Reasoning:   Hardest reasoning step — must understand that MailerLite segment must exist before Make scenario can reference its ID; wrong ordering causes cascading failures; runs once per build so Opus cost is justified

payload_generation:
  Use case:    Generate exact API payloads for eac

## Quality checklist

Before you're done, verify:

- [ ] Tokenization code runs and produces output for all 5 inputs
- [ ] Token counts compared across 2 tokenizers (tiktoken + Anthropic)
- [ ] Parameter experiments run for all 4 experiments
- [ ] Each experiment has documented observations — not just raw output
- [ ] Analysis explains the parameter–output relationship ("temperature controls X because Y," not just "temperature changed the output")
- [ ] Cost calculations completed: per-call costs, multilingual cost multiplier, context budget
- [ ] Capstone capability document has at least 3 capabilities with parameter rationale and model tier choice
- [ ] Capstone cost estimate is realistic given your expected usage scale
- [ ] Code is clean enough that a peer could run it and understand what each experiment tests

## Stretch goals

1. **Top-p experiment:** Run the creative prompt at top_p=0.1, 0.5, 0.9 (with temperature=1.0). Compare output diversity with your temperature results. When would you use top_p instead of temperature?
2. **CLI tool:** Build a command-line tool that accepts a prompt and parameter values as arguments and prints the response with usage stats and cost estimate.
3. **Multilingual token analysis:** Tokenize "Artificial intelligence is changing software development" in 5 languages. Plot the token-to-character ratio. What pattern emerges?
4. **"Lost in the middle" test:** Create a ~5000-token prompt with a specific fact buried in the middle. Ask the model to retrieve it. Move the fact to the beginning and end — does accuracy change? (Preview of why RAG matters in M4.)

## What I learned

Write 3–6 bullets in your own words, covering:

- I learned about how different system prompts and input prompts affect the results tremendously.
- Make sure which models to use for what, higher cost for higher reasoning.
- I defined what my capstone will do, what its requirements are and the cost prediction for it.

_This cell is required — `check-notebook.py` enforces it. It's also what makes this notebook a portfolio artifact when you publish to GitHub._